# Barbados Traffic (Zindi) — Pipeline `submission_PHYSICS_ENGINEER.csv`

Version portable du notebook de compétition : la logique est identique, seuls les chemins `/content/drive/...` ont été remplacés par `DATA_DIR` / `FEATURES_DIR` / `SUBMISSION_DIR`.

| Étape | Rôle | Besoin |
|---|---|---|
| 0 | Configuration des chemins | — |
| 1-2 | Extraction des features vidéo → `all_features_EXPERT.csv` | GPU + accès au bucket GCS, plusieurs heures |
| 3 | Feature engineering + LightGBM × 10 seeds → soumission | CPU, quelques minutes |

> Si `input/all_features_EXPERT.csv` est présent, passe directement à l'étape 3.

## 0. Configuration

In [ ]:
# ============================================================
# CONFIGURATION DES CHEMINS (seule cellule à adapter)
# ============================================================
import os

try:
    from google.colab import drive          # Exécution dans Colab
    drive.mount('/content/drive')
    ROOT = '/content/drive/MyDrive/zindi-barbados-traffic'
except ImportError:                          # Exécution en local
    ROOT = os.path.abspath('..')             # racine du dépôt (le notebook est dans notebooks/)

DATA_DIR       = os.path.join(ROOT, 'input')        # Train.csv, TestInputSegments.csv, SampleSubmission.csv
FEATURES_DIR   = os.path.join(ROOT, 'input')   # all_features_EXPERT.csv
SUBMISSION_DIR = os.path.join(ROOT, 'output')
os.makedirs(FEATURES_DIR, exist_ok=True)
os.makedirs(SUBMISSION_DIR, exist_ok=True)

print('DATA_DIR      :', DATA_DIR)
print('FEATURES_DIR  :', FEATURES_DIR)
print('SUBMISSION_DIR:', SUBMISSION_DIR)

## 1. Extracteur de features vidéo *(optionnel si `all_features_EXPERT.csv` est présent)*
Faster R-CNN ResNet50-FPN v2 (COCO) sur 5 frames 320×180 + flux optique Farneback.

In [ ]:
# Uniquement pour l'extraction depuis le bucket (étapes 1-2)
!gcloud auth login
!gcloud auth application-default login
from google.cloud import storage
client = storage.Client(project="brb-traffic")

In [ ]:
# --- CELLULE 1 : CONFIGURATION ET FONCTION D'EXTRACTION EXPERTE ---
import os
import torch
import cv2
import numpy as np
import pandas as pd
from google.cloud import storage
from tqdm import tqdm
from torchvision.models.detection import fasterrcnn_resnet50_fpn_v2, FasterRCNN_ResNet50_FPN_V2_Weights
from torchvision.transforms import functional as F
from PIL import Image

# 1. Chargement du Modèle (GPU si dispo)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✅ Utilisation du device : {device}")

weights = FasterRCNN_ResNet50_FPN_V2_Weights.DEFAULT
model = fasterrcnn_resnet50_fpn_v2(weights=weights).to(device)
model.eval()

# Classes COCO pertinentes
COCO_TO_CUSTOM = {1: 'person', 3: 'car', 8: 'truck', 6: 'bus'}

# 2. Fonction d'Extraction Optimisée (Vitesse + Features Expertes)
def extract_features_from_video(video_file):
    cap = cv2.VideoCapture(video_file)
    if not cap.isOpened(): return None

    vehicle_counts = {'car': 0, 'truck': 0, 'bus': 0, 'person': 0}
    flow_speeds = []

    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total_frames <= 0: total_frames = 300 # Sécurité

    # STRATÉGIE SNAPSHOT : On ne regarde que 5 images réparties sur la vidéo
    frame_indices = np.linspace(0, total_frames - 2, 5, dtype=int)

    prev_gray = None

    for frame_idx in frame_indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
        ret, frame = cap.read()
        if not ret: break

        # A. Optimisation Vitesse : Redimensionner en 320x180
        frame_small = cv2.resize(frame, (320, 180))

        # B. Détection (FasterRCNN)
        img = Image.fromarray(cv2.cvtColor(frame_small, cv2.COLOR_BGR2RGB))
        img_tensor = F.to_tensor(img).unsqueeze(0).to(device)

        with torch.no_grad():
            preds = model(img_tensor)[0]

        for label, score in zip(preds['labels'], preds['scores']):
            # Seuil 0.4 suffisant pour basse résolution
            if score > 0.4 and label.item() in COCO_TO_CUSTOM:
                vehicle_counts[COCO_TO_CUSTOM[label.item()]] += 1

        # C. Flux Optique (Vitesse)
        ret2, frame_next = cap.read()
        if ret2:
            frame_next_small = cv2.resize(frame_next, (320, 180))
            gray_curr = cv2.cvtColor(frame_small, cv2.COLOR_BGR2GRAY)
            gray_next = cv2.cvtColor(frame_next_small, cv2.COLOR_BGR2GRAY)

            flow = cv2.calcOpticalFlowFarneback(gray_curr, gray_next, None, 0.5, 3, 15, 3, 5, 1.2, 0)
            magnitudes = np.sqrt(flow[..., 0] ** 2 + flow[..., 1] ** 2)
            flow_speeds.append(np.mean(magnitudes))

    cap.release()

    divider = len(frame_indices) if len(frame_indices) > 0 else 1

    # --- CALCUL DES FEATURES ---
    avg_speed = np.mean(flow_speeds) if flow_speeds else 0
    total_vehicles = sum(vehicle_counts.values()) / divider

    # Expert 1 : Variation de vitesse (Indique stop & go)
    std_speed = np.std(flow_speeds) if flow_speeds else 0

    # Expert 2 : Indice de Congestion (Densité / Vitesse)
    congestion_index = total_vehicles / (avg_speed + 0.1)

    return {
        'num_cars': vehicle_counts['car'] / divider,
        'num_buses': vehicle_counts['bus'] / divider,
        'num_trucks': vehicle_counts['truck'] / divider,
        'num_peds': vehicle_counts['person'] / divider,
        'avg_speed': avg_speed,
        'density': total_vehicles,
        'queue_length_proxy': max(vehicle_counts['car'], vehicle_counts['bus']) / divider,
        'std_speed': std_speed,
        'congestion_index': congestion_index
    }
print("✅ Configuration terminée.")

## 2. Extraction → `all_features_EXPERT.csv` *(optionnel)*
À relancer jusqu'à `Reste à traiter : 0 Tests | 0 Trains`.

In [ ]:
# --- CELLULE 2 : TRAITEMENT PRIORITAIRE (TEST INPUT FIRST) ---
import os
import pandas as pd
from google.cloud import storage

# --- CONFIGURATION ---
client = storage.Client(project="brb-traffic")
bucket_name = 'brb-traffic'
drive_save_path = FEATURES_DIR
os.makedirs(drive_save_path, exist_ok=True)

features_file = os.path.join(drive_save_path, 'all_features_EXPERT.csv') # Fichier EXPERT
processed_log = os.path.join(drive_save_path, 'processed_log.txt')
local_video_folder = 'videos_temp'
os.makedirs(local_video_folder, exist_ok=True)

# --- CHARGEMENT ROBUSTE ---
try:
    train_df = pd.read_csv(os.path.join(DATA_DIR, 'Train.csv'))
    test_df = pd.read_csv(os.path.join(DATA_DIR, 'TestInputSegments.csv'))
except:
    train_df = pd.read_csv('Train.csv')
    test_df = pd.read_csv('TestInputSegments.csv')

def extract_path(v): return f"{v.split('/')[-1].split('_')[0]}/{v.split('/')[-1]}"
train_df['videos_path'] = train_df['videos'].apply(extract_path)
test_df['videos_path'] = test_df['videos'].apply(extract_path)
train_df['dataset'] = 'train'; test_df['dataset'] = 'test'
full_df = pd.concat([train_df, test_df], ignore_index=True)

# --- FILTRAGE ET PRIORISATION ---
def get_done():
    if not os.path.exists(processed_log): return set()
    with open(processed_log, 'r') as f: return set(l.strip() for l in f)

processed = get_done()
remaining = full_df[~full_df['videos_path'].isin(processed)]

# On sépare Test et Train
test_remaining = remaining[remaining['dataset'] == 'test']
train_remaining = remaining[remaining['dataset'] == 'train']

print(f"📊 Reste à traiter : {len(test_remaining)} Tests (URGENT) | {len(train_remaining)} Trains")

# STRATÉGIE : Si il reste du Test, on ne fait QUE ça. Sinon, on fait du Train.
if len(test_remaining) > 0:
    print("⚠️ MODE PRIORITÉ : Traitement du Test Input en cours...")
    videos_to_process = test_remaining # Pas de mélange, on veut finir les séries
else:
    print("✅ Test Input fini ! On traite le Train pour améliorer le modèle.")
    videos_to_process = train_remaining.sample(frac=1, random_state=42)

BATCH_LIMIT = 5000
count = 0

print(f"🚀 Démarrage du batch ({len(videos_to_process)} vidéos)...")

for index, row in tqdm(videos_to_process.iterrows(), total=len(videos_to_process)):
    if count >= BATCH_LIMIT: break

    path = row['videos_path']
    local = os.path.join(local_video_folder, os.path.basename(path))

    try:
        # Téléchargement
        blob = client.bucket(bucket_name).blob(path)
        blob.download_to_filename(local)

        # Extraction
        feats = extract_features_from_video(local)

        if feats:
            feats.update({
                'videos_path': path, 'time_segment_id': row['time_segment_id'],
                'view_label': row['view_label'], 'date': row['date'],
                'signaling': row.get('signaling', 'unknown'),
                'video_time': row.get('video_time', ''), 'dataset': row['dataset']
            })
            # Sauvegarde directe
            pd.DataFrame([feats]).to_csv(features_file, mode='a', header=not os.path.exists(features_file), index=False)

        # Marquage comme fait
        with open(processed_log, 'a') as f: f.write(f"{path}\n")
        if os.path.exists(local): os.remove(local)
        count += 1

    except Exception as e:
        # On marque quand même comme fait (pour éviter boucle 404)
        with open(processed_log, 'a') as f: f.write(f"{path}\n")
        if os.path.exists(local): os.remove(local)
        continue

print("🏁 Batch terminé.")

## 3. Modèle « L'Ingénieur Trafic » → `submission_PHYSICS_ENGINEER.csv`

In [ ]:
# --- CELLULE 90 : L'INGÉNIEUR TRAFIC (PHYSIQUE & CYCLES) ---
import pandas as pd
import numpy as np
import os
from lightgbm import LGBMClassifier
from sklearn.preprocessing import RobustScaler

print("👷 DÉMARRAGE DE L'INGÉNIEUR TRAFIC (PHYSICS BOOST)...")

# ============================================================
# 1. PRÉPARATION & NETTOYAGE
# ============================================================
try:
    df_features = pd.read_csv('all_features_EXPERT.csv')
    train_original = pd.read_csv('Train.csv')
    ss = pd.read_csv('SampleSubmission.csv')
except:
    base_path = FEATURES_DIR
    df_features = pd.read_csv(os.path.join(base_path, 'all_features_EXPERT.csv'))
    train_original = pd.read_csv(os.path.join(DATA_DIR, 'Train.csv'))
    ss = pd.read_csv(os.path.join(DATA_DIR, 'SampleSubmission.csv'))

def get_clean_id(x):
    try: return int(str(x).split('_')[-5])
    except: return int(x)
df_features['clean_seg_id'] = df_features.apply(lambda r: get_clean_id(r.get('time_segment_id', r.get('videos_path'))), axis=1)
df_features.loc[df_features['num_cars'] == 0, ['density', 'congestion_index']] = 0

# GLOBAL STATS (La base solide)
global_stats = df_features.groupby('clean_seg_id').agg({
    'density': ['mean', 'sum'], 'avg_speed': 'mean', 'congestion_index': 'max'
}).reset_index()
global_stats.columns = ['clean_seg_id', 'glob_dens_mean', 'glob_dens_sum', 'glob_spd_mean', 'glob_cong_max']
df_features = df_features.merge(global_stats, on='clean_seg_id', how='left')

# ============================================================
# 2. FEATURE ENGINEERING : LA PHYSIQUE (NOUVEAU !)
# ============================================================
print("   🧪 Injection des formules physiques...")

# A. FLUX & INTENSITÉ (Physique des fluides)
# On ajoute +1 au diviseur pour éviter la division par zéro
df_features['phys_flow'] = df_features['glob_dens_mean'] * df_features['glob_spd_mean'] # Débit
df_features['phys_jam'] = df_features['glob_dens_mean'] / (df_features['glob_spd_mean'] + 1) # Intensité du bouchon

# B. CYCLICITÉ TEMPORELLE (Mathématiques)
# Transforme l'heure en cercle (23h touche 0h)
df_features['video_time'] = pd.to_datetime(df_features['video_time'])
df_features['hour'] = df_features['video_time'].dt.hour
df_features['minute'] = df_features['video_time'].dt.minute

df_features['hour_sin'] = np.sin(2 * np.pi * df_features['hour']/24)
df_features['hour_cos'] = np.cos(2 * np.pi * df_features['hour']/24)

# On garde le tri temporel pour les lags
df_features = df_features.sort_values(['view_label', 'clean_seg_id'])

# LAGS (On inclut les nouvelles features physiques !)
cols_to_lag = ['num_cars', 'avg_speed', 'density', 'phys_flow', 'phys_jam', 'glob_dens_mean', 'glob_spd_mean']
lag_dfs = []
for lag in [1, 2, 3]:
    shifted = df_features.groupby('view_label')[cols_to_lag].shift(lag)
    shifted.columns = [f'{col}_lag{lag}' for col in cols_to_lag]
    lag_dfs.append(shifted)
df_features = pd.concat([df_features] + lag_dfs, axis=1).fillna(0)

# Normalisation
scaler = RobustScaler()
# On sélectionne toutes les colonnes numériques utiles
numeric_cols = df_features.select_dtypes(include=[np.number]).columns
exclude_cols = ['clean_seg_id', 'hour', 'minute'] # On ne normalise pas les IDs ou le temps brut
cols_to_scale = [c for c in numeric_cols if c not in exclude_cols]
df_features[cols_to_scale] = scaler.fit_transform(df_features[cols_to_scale])

# ============================================================
# 3. PRÉPARATION TRAIN/TEST
# ============================================================
train_labels = pd.read_csv(os.path.join(DATA_DIR, 'Train.csv'))
train_full = df_features[df_features['dataset'] == 'train'].merge(
    train_labels[['time_segment_id', 'view_label', 'congestion_enter_rating']],
    left_on=['clean_seg_id', 'view_label'], right_on=['time_segment_id', 'view_label'], how='inner'
)
train_full = train_full.sort_values(['view_label', 'clean_seg_id'])
train_full['target_future'] = train_full.groupby('view_label')['congestion_enter_rating'].shift(-3)
train_ready = train_full.dropna(subset=['target_future']).copy()
test_ready = df_features[df_features['dataset'] == 'test'].copy()
test_ready = test_ready.sort_values(['view_label', 'clean_seg_id'])

drop_cols = ['dataset', 'videos_path', 'video_time', 'date', 'congestion_enter_rating', 'target_future', 'y_target', 'signaling', 'view_label']
feature_cols = [c for c in train_ready.columns if c not in drop_cols and c in test_ready.columns and train_ready[c].dtype in [np.float64, np.int64]]

# ============================================================
# 4. ENTRAÎNEMENT (POIDS FINAL CUT)
# ============================================================
# On garde tes poids gagnants : Light=4.2, Moderate=3.1
weights_final = {'free flowing': 1.0, 'light delay': 4.2, 'moderate delay': 3.1, 'heavy delay': 4.5}
print(f"🚀 Entraînement Physique ({len(feature_cols)} features)...")

params = {
    'n_estimators': 3500, 'learning_rate': 0.012, 'num_leaves': 35,
    'subsample': 0.8, 'colsample_bytree': 0.7, 'class_weight': weights_final,
    'n_jobs': -1, 'verbose': -1
}

X = train_ready[feature_cols]
y = train_ready['target_future']
X_test = test_ready[feature_cols]

SEEDS = [42, 2025, 777, 101, 888, 999, 123, 456, 1000, 555] # 10 seeds standard
final_probs = np.zeros((len(test_ready), 4))
classes_order = None

for i, seed in enumerate(SEEDS):
    print(f"   🌱 Seed {i+1}/10...", end='\r')
    params['random_state'] = seed
    clf = LGBMClassifier(**params)
    clf.fit(X, y)
    if classes_order is None: classes_order = clf.classes_
    final_probs += clf.predict_proba(X_test) / len(SEEDS)

# ============================================================
# 5. LISSAGE TEMPOREL (CELUI QUI A DONNÉ 0.626)
# ============================================================
print("\n🌊 Lissage Temporel...")
test_ready['prob_0'] = final_probs[:, 0]
test_ready['prob_1'] = final_probs[:, 1]
test_ready['prob_2'] = final_probs[:, 2]
test_ready['prob_3'] = final_probs[:, 3]

# Rolling mean (Window 3)
rolling_probs = test_ready.groupby('view_label')[['prob_0', 'prob_1', 'prob_2', 'prob_3']].transform(lambda x: x.rolling(window=3, center=True, min_periods=1).mean())

# Mix 60% Brut / 40% Lissée
final_probs_smoothed = (test_ready[['prob_0', 'prob_1', 'prob_2', 'prob_3']].values * 0.6) + (rolling_probs.values * 0.4)

# ============================================================
# 6. SOUMISSION
# ============================================================
print("🔧 Calibration...")
# Ordre : Free, Heavy, Light, Moderate
calib_dict = {'free flowing': 0.95, 'light delay': 1.15, 'moderate delay': 1.05, 'heavy delay': 1.0}
calibration_vector = np.array([calib_dict[c] for c in classes_order])
final_probs_calibrated = final_probs_smoothed * calibration_vector

final_preds_idx = np.argmax(final_probs_calibrated, axis=1)
final_preds_str = [classes_order[i] for i in final_preds_idx]

test_ready['final_pred'] = final_preds_str
submission_map = dict(zip(test_ready['clean_seg_id'].astype(str) + '_' + test_ready['view_label'], test_ready['final_pred']))

knowledge_base = {}
for (idx, row), pred in zip(test_ready.iterrows(), final_preds_str):
    knowledge_base.setdefault(row['view_label'], {})[row['clean_seg_id']] = pred

submission_rows = []
for _, row in ss.iterrows():
    row_id = row['ID']
    val = "free flowing"
    try:
        parts = row_id.split('_')
        target_id = int(parts[2])
        view = next((v for v in ["Norman Niles #1", "Norman Niles #2", "Norman Niles #3", "Norman Niles #4"] if v in row_id), "Unknown")
        key = str(target_id) + '_' + view
        if key in submission_map: val = submission_map[key]
        else:
            if view in knowledge_base:
                candidates = [pid for pid in knowledge_base[view].keys() if pid <= target_id]
                if candidates:
                    best_source = max(candidates)
                    if (target_id - best_source) < 15:
                        val = knowledge_base[view][best_source]
    except: pass
    submission_rows.append({'ID': row_id, 'Target': val, 'Target_Accuracy': val})

sub_df = pd.DataFrame(submission_rows)
filename = os.path.join(SUBMISSION_DIR, 'submission_PHYSICS_ENGINEER.csv')
sub_df.to_csv(filename, index=False)

print(f"\n📊 RÉPARTITION FINALE ({filename}) :")
print(sub_df['Target'].value_counts())

## 4. Vérification

In [ ]:
# --- VÉRIFICATION ---
import pandas as pd
new = pd.read_csv(os.path.join(SUBMISSION_DIR, 'submission_PHYSICS_ENGINEER.csv'))
print(new.shape)                      # attendu : (880, 3)
print(new['Target'].value_counts())   # attendu : free 550 | moderate 165 | heavy 105 | light 60